# UrbanFloodBench: Hybrid Baseline + Correction (Inference Only)

This notebook is a trimmed version of `UrbanFloodBench_HybridBaseline_Correction.ipynb`.

**What it does**
- Loads graph + dynamic arrays
- Computes (or loads) normalization stats
- Fits (or loads) the ARX ridge baseline using the known best `lam=0.1`
- Loads your trained correction checkpoints: `corr_model_1.pt`, `corr_model_2.pt`
- Runs test inference and writes a submission CSV

**What it does NOT do**
- Any lambda sweeps
- Any correction-model training (the slow part)


In [1]:
# -----------------------------
# 0) Setup
# -----------------------------
import os, gc, math, time
from pathlib import Path
from glob import glob
from collections import defaultdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

# Repro
SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[env] device:", device)

# -----------------------------
# Paths (edit if your folder names differ)
# -----------------------------
BASE_PATH = Path("data")  # contains Model_1/ and Model_2/
SUB_PATH  = Path("early_data/sample_submission.csv")

# Your trained correction checkpoints
CKPT_CORR_1 = Path("corr_model_1.pt")
CKPT_CORR_2 = Path("corr_model_2.pt")

# Cache directory for memmaps + stats/baseline weights
CACHE_DIR = Path("data/_cache_infer")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN      = int(os.environ.get("FLOOD_HIDDEN", 64))
CORR_SCALE  = float(os.environ.get("FLOOD_CORR_SCALE", 0.2))   # tanh-bounded correction scale (in normalized units)
TF_START    = float(os.environ.get("FLOOD_TF_START", 1.0))     # teacher forcing prob at start
TF_END      = float(os.environ.get("FLOOD_TF_END", 0.3))       # teacher forcing prob near end
TF_WARMUP   = int(os.environ.get("FLOOD_TF_WARMUP", 8000))     # steps to anneal TF prob
# -----------------------------
# Inference hyperparams (must match training)
# -----------------------------
LOOKBACK = 10          # t=0..9 are observed; predict t>=10
CLIP_STD = 3.0         # clamp standardized predictions for stability

# Baseline ARX settings (must match the baseline you used to train correction)
USE_RAIN_MEMORY = True
RAIN_CUM_CLIP = 25.0
RAIN_EMA_ALPHA = 0.15

# Correction net settings (must match the checkpoint)
CORR_HIDDEN = 64
TYPE_EMB_DIM = 8
NEIGH_K = 8
ROUNDS = 2

# Known best ridge lambda from your sweep logs
BASELINE_LAM = 0.1

# Per-event standardized RMSE normalization constants (from the original notebook)
STD_DEV_DICT = {
    (1, 1): 0.020705781,
    (1, 2): 0.005096739,
    (2, 1): 0.234103457,
    (2, 2): 0.099197844,
}


[env] device: cuda


## Metrics helpers (only used if you sanity-check on train/val)

In [2]:
from collections import defaultdict

class ParticipantVisibleError(Exception):
    pass

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def standardized_rmse(y_true, y_pred, std_dev):
    if std_dev == 0 or np.isnan(std_dev):
        return np.nan
    return rmse(y_true, y_pred) / std_dev

def calculate_event_std_rmse_from_arrays(node_types, node_ids, y_true, y_pred, std_devs_by_type):
    node_types = np.asarray(node_types)
    node_ids = np.asarray(node_ids)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask_1d = node_types == 1
    mask_2d = node_types == 2

    unique_1d_nodes = np.unique(node_ids[mask_1d]) if mask_1d.any() else np.array([])
    unique_2d_nodes = np.unique(node_ids[mask_2d]) if mask_2d.any() else np.array([])

    std_rmse_1d_list = []
    for node_id in unique_1d_nodes:
        m = (node_ids == node_id) & (node_types == 1)
        if np.sum(m) > 1:
            v = standardized_rmse(y_true[m], y_pred[m], std_devs_by_type.get(1, np.nan))
            if np.isfinite(v):
                std_rmse_1d_list.append(v)

    std_rmse_2d_list = []
    for node_id in unique_2d_nodes:
        m = (node_ids == node_id) & (node_types == 2)
        if np.sum(m) > 1:
            v = standardized_rmse(y_true[m], y_pred[m], std_devs_by_type.get(2, np.nan))
            if np.isfinite(v):
                std_rmse_2d_list.append(v)

    avg_1d = np.mean(std_rmse_1d_list) if std_rmse_1d_list else np.nan
    avg_2d = np.mean(std_rmse_2d_list) if std_rmse_2d_list else np.nan
    valid = [x for x in [avg_1d, avg_2d] if np.isfinite(x)]
    return float(np.mean(valid)) if valid else np.nan

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "row_id") -> float:
    required_cols = ["model_id", "event_id", "node_type", "node_id", "water_level"]
    key_cols = ["model_id", "event_id", "node_type", "node_id"]

    missing_cols = [c for c in required_cols if c not in submission.columns]
    if missing_cols:
        raise ParticipantVisibleError(f"Submission missing required column(s): {', '.join(missing_cols)}")

    cols_to_check = [c for c in submission.columns if c != row_id_column_name]
    extra_cols = [c for c in cols_to_check if c not in required_cols]
    if extra_cols:
        raise ParticipantVisibleError(f"Submission has unexpected column(s): {', '.join(extra_cols)}")

    if len(submission) != len(solution):
        raise ParticipantVisibleError(f"Submission has {len(submission)} rows but expected {len(solution)} rows")


    if submission["water_level"].isna().any():
        raise ParticipantVisibleError("Submission contains NaN in water_level")

    if not np.isfinite(submission["water_level"]).all():
        raise ParticipantVisibleError("Submission contains non-finite values in water_level")

    solution_work = solution[key_cols + ["water_level"]].copy().sort_values(key_cols).reset_index(drop=True)
    submission_work = submission[key_cols + ["water_level"]].copy().sort_values(key_cols).reset_index(drop=True)

    for col in key_cols:
        if not solution_work[col].equals(submission_work[col]):
            raise ParticipantVisibleError("Submission does not match solution structure")

    keys = list(zip(solution_work["model_id"].values, solution_work["event_id"].values))
    boundaries = []
    prev = None
    s = 0
    for i, k in enumerate(keys):
        if k != prev:
            if prev is not None:
                boundaries.append((prev, s, i))
            prev = k
            s = i
    if prev is not None:
        boundaries.append((prev, s, len(keys)))

    model_event_scores = defaultdict(list)
    for (model_id, _event_id), a, b in boundaries:
        node_types = solution_work["node_type"].values[a:b]
        node_ids = solution_work["node_id"].values[a:b]
        y_true = solution_work["water_level"].values[a:b]
        y_pred = submission_work["water_level"].values[a:b]
        std_devs = {1: STD_DEV_DICT[(model_id, 1)], 2: STD_DEV_DICT[(model_id, 2)]}
        v = calculate_event_std_rmse_from_arrays(node_types, node_ids, y_true, y_pred, std_devs)
        if np.isfinite(v):
            model_event_scores[model_id].append(v)

    model_scores = []
    for model_id in sorted(model_event_scores.keys()):
        vals = model_event_scores[model_id]
        if not vals:
            raise ParticipantVisibleError(f"No valid events for model_id {model_id}")
        model_scores.append(float(np.mean(vals)))
    if len(model_scores) != 2:
        raise ParticipantVisibleError("Expected 2 model scores")
    return float(np.mean(model_scores))


## Graph static loading (Model_1 and Model_2)

In [3]:

# =============================
# 2) File resolution + loading helpers
# =============================
def resolve_event_file(event_dir, fname):
    candidates = [fname, f"test_{fname}"]
    for c in candidates:
        p = os.path.join(event_dir, c)
        if os.path.exists(p):
            return p
    return None

def resolve_static_file(base_dir, fname):
    candidates = [fname, f"test_{fname}"]
    for c in candidates:
        p = os.path.join(base_dir, c)
        if os.path.exists(p):
            return p
    return None

def find_col(df, keys, contains=None):
    lower = {c.lower(): c for c in df.columns}
    for k in keys:
        if k.lower() in lower:
            return lower[k.lower()]
    if contains:
        for c in df.columns:
            if contains.lower() in c.lower():
                return c
    return None

def ensure_time_index(df, node_col):
    t_col = find_col(df, ["t", "time", "time_idx", "timestep"], None)
    if t_col is None:
        df = df.copy()
        df["t"] = df.groupby(node_col).cumcount()
        t_col = "t"
    else:
        if t_col != "t":
            df = df.rename(columns={t_col: "t"})
            t_col = "t"
    return df, t_col

def list_events(model_id, split):
    base = os.path.join(BASE_PATH, f"Model_{model_id}", split)
    if not os.path.exists(base):
        return []
    out = []
    for name in os.listdir(base):
        if "event" in name:
            try:
                out.append(int(name.split("_")[-1]))
            except Exception:
                pass
    return sorted(out)

def load_static_nodes(model_id, node_type):
    train_dir = os.path.join(BASE_PATH, f"Model_{model_id}", "train")
    test_dir  = os.path.join(BASE_PATH, f"Model_{model_id}", "test")
    fname = "1d_nodes_static.csv" if node_type == 1 else "2d_nodes_static.csv"
    fpath = resolve_static_file(train_dir, fname) or resolve_static_file(test_dir, fname)
    if fpath is None:
        return None
    df = pd.read_csv(fpath)
    node_col = find_col(df, ["node_idx", "node_id"], None)
    if node_col is None:
        return None
    df = df.rename(columns={node_col: "node_idx"})
    return df

def load_edge_index(model_id, node_type):
    train_dir = os.path.join(BASE_PATH, f"Model_{model_id}", "train")
    test_dir  = os.path.join(BASE_PATH, f"Model_{model_id}", "test")
    fname = "1d_edge_index.csv" if node_type == 1 else "2d_edge_index.csv"
    fpath = resolve_static_file(train_dir, fname) or resolve_static_file(test_dir, fname)
    if fpath is None:
        return None
    df = pd.read_csv(fpath)
    c0 = find_col(df, ["from", "src", "u", "node_u", "node_from"], None)
    c1 = find_col(df, ["to", "dst", "v", "node_v", "node_to"], None)
    if c0 is None or c1 is None:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) >= 2:
            c0, c1 = num_cols[0], num_cols[1]
        else:
            return None
    return df[[c0, c1]].astype(int)

def load_connections(model_id):
    train_dir = os.path.join(BASE_PATH, f"Model_{model_id}", "train")
    test_dir  = os.path.join(BASE_PATH, f"Model_{model_id}", "test")
    fname = "1d2d_connections.csv"
    fpath = resolve_static_file(train_dir, fname) or resolve_static_file(test_dir, fname)
    if fpath is None:
        return None
    df = pd.read_csv(fpath)
    cols_1d = [c for c in df.columns if "1d" in c.lower()]
    cols_2d = [c for c in df.columns if "2d" in c.lower()]
    if not cols_1d or not cols_2d:
        return None
    col_1d = cols_1d[0]
    col_2d = cols_2d[0]
    return df[[col_1d, col_2d]].dropna().astype(int)

def load_dynamic_nodes(model_id, event_id, split, node_type):
    event_dir = os.path.join(BASE_PATH, f"Model_{model_id}", split, f"event_{event_id}")
    fname = "1d_nodes_dynamic_all.csv" if node_type == 1 else "2d_nodes_dynamic_all.csv"
    fpath = resolve_event_file(event_dir, fname)
    if fpath is None:
        return None
    df = pd.read_csv(fpath)
    node_col = find_col(df, ["node_idx", "node_id"], None)
    if node_col is None:
        return None
    water_col = find_col(df, ["water_level"], "water")
    rain_col  = find_col(df, ["rainfall"], "rain") if node_type == 2 else None

    cols = [node_col]
    if water_col: cols.append(water_col)
    if rain_col: cols.append(rain_col)

    df = df[cols]
    df, _ = ensure_time_index(df, node_col)

    out = df.rename(columns={node_col: "node_idx"})
    if water_col and water_col != "water_level":
        out = out.rename(columns={water_col: "water_level"})
    if rain_col and rain_col != "rainfall":
        out = out.rename(columns={rain_col: "rainfall"})
    if "rainfall" not in out.columns:
        out["rainfall"] = 0.0

    return out

def split_events_by_event(events, val_frac, seed):
    events = list(events)
    if len(events) <= 1:
        return events, []
    rng = np.random.RandomState(seed)
    rng.shuffle(events)
    n_val = max(1, int(len(events) * val_frac))
    return events[n_val:], events[:n_val]


In [4]:

# =============================
# 3) Build node maps + static feature tensors + graph edge_index tensors
# =============================
def select_static_cols(df, node_type):
    if df is None or df.empty:
        return []
    if node_type == 1:
        preferred = ["depth", "invert_elevation", "surface_elevation", "base_area", "slope", "roughness"]
    else:
        preferred = ["min_elevation", "elevation", "roughness", "area", "flow_accumulation", "slope"]
    cols = [c for c in preferred if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if cols:
        return cols
    cols = [c for c in df.columns if c != "node_idx" and pd.api.types.is_numeric_dtype(df[c])]
    return cols

def build_static_norm(df, cols):
    if df is None or df.empty or not cols:
        return None, None, None
    X = df[cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd[sd == 0] = 1.0
    Xn = (X - mu) / sd
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=0.0, neginf=0.0)
    return Xn, mu, sd

def build_index_map(node_ids):
    node_ids = np.asarray(node_ids).astype(int)
    node_ids_sorted = np.sort(node_ids)
    idx = {int(n): i for i, n in enumerate(node_ids_sorted)}
    inv = node_ids_sorted
    return idx, inv

def remap_edges(df_edges, idx_map):
    if df_edges is None or df_edges.empty:
        return None
    src = df_edges.iloc[:, 0].values.astype(int)
    dst = df_edges.iloc[:, 1].values.astype(int)
    m = []
    for a, b in zip(src, dst):
        if a in idx_map and b in idx_map:
            m.append((idx_map[a], idx_map[b]))
    if not m:
        return None
    arr = np.array(m, dtype=np.int64).T  # [2, E]
    return arr

graph_cache = {}

for model_id in [1, 2]:
    s1 = load_static_nodes(model_id, 1)
    s2 = load_static_nodes(model_id, 2)
    if s1 is None or s2 is None:
        raise RuntimeError(f"Missing static nodes for model {model_id}")

    idx1, inv1 = build_index_map(s1["node_idx"].values)
    idx2, inv2 = build_index_map(s2["node_idx"].values)

    cols1 = select_static_cols(s1, 1)
    cols2 = select_static_cols(s2, 2)

    X1n, _, _ = build_static_norm(s1, cols1)
    X2n, _, _ = build_static_norm(s2, cols2)

    N1 = len(inv1)
    N2 = len(inv2)

    S1 = 0 if X1n is None else X1n.shape[1]
    S2 = 0 if X2n is None else X2n.shape[1]
    S_total = S1 + S2

    static_all = np.zeros((N1 + N2, S_total), dtype=np.float32)
    if S1:
        static_all[:N1, :S1] = X1n
    if S2:
        static_all[N1:, S1:] = X2n

    e1 = load_edge_index(model_id, 1)
    e2 = load_edge_index(model_id, 2)
    conn = load_connections(model_id)

    e1_m = remap_edges(e1, idx1) if e1 is not None else None
    e2_m = remap_edges(e2, idx2) if e2 is not None else None
    if e2_m is not None:
        e2_m = e2_m.copy()
        e2_m[0] += N1
        e2_m[1] += N1

    e12 = None
    if conn is not None and not conn.empty:
        c1 = conn.iloc[:, 0].values.astype(int)
        c2 = conn.iloc[:, 1].values.astype(int)
        pairs = []
        for a, b in zip(c1, c2):
            if a in idx1 and b in idx2:
                ia = idx1[a]
                ib = idx2[b] + N1
                pairs.append((ia, ib))
                pairs.append((ib, ia))
        if pairs:
            e12 = np.array(pairs, dtype=np.int64).T

    def make_undirected(edge):
        if edge is None:
            return None
        src, dst = edge
        rev = np.stack([dst, src], axis=0)
        return np.concatenate([edge, rev], axis=1)

    e1_u = make_undirected(e1_m)
    e2_u = make_undirected(e2_m)
    e12_u = make_undirected(e12)

    graph_cache[model_id] = dict(
        N1=N1, N2=N2, N=N1+N2,
        idx1=idx1, idx2=idx2, inv1=inv1, inv2=inv2,
        cols1=cols1, cols2=cols2,
        static_all=static_all,
        edges_1d=e1_u,
        edges_2d=e2_u,
        edges_cross=e12_u,
        conn_df=conn
    )

    print(f"[model {model_id}] N1={N1} N2={N2} static_dim={S_total} edges_1d={0 if e1_u is None else e1_u.shape[1]} edges_2d={0 if e2_u is None else e2_u.shape[1]} cross={0 if e12_u is None else e12_u.shape[1]}")


[model 1] N1=17 N2=3716 static_dim=9 edges_1d=32 edges_2d=7432 cross=64
[model 2] N1=198 N2=4299 static_dim=9 edges_1d=394 edges_2d=8598 cross=788


## Rain mapping + dynamic event array loading

In [5]:

# =============================
# 4) 1D rainfall from 2D via connection map
# =============================
def build_conn_lists(model_id):
    g = graph_cache[model_id]
    conn = g["conn_df"]
    N1 = g["N1"]
    idx1 = g["idx1"]
    idx2 = g["idx2"]
    if conn is None or conn.empty:
        return [None] * N1

    c1 = conn.iloc[:, 0].values.astype(int)
    c2 = conn.iloc[:, 1].values.astype(int)

    lst = [[] for _ in range(N1)]
    for a, b in zip(c1, c2):
        if a in idx1 and b in idx2:
            ia = idx1[a]
            ib = idx2[b]
            lst[ia].append(ib)
    out = [x if x else None for x in lst]
    return out

conn_lists = {m: build_conn_lists(m) for m in [1, 2]}

def build_rain_1d_from_2d(rain_2d_TN2, conn_list, global_rain_T):
    T, N2 = rain_2d_TN2.shape
    N1 = len(conn_list)
    rain_1d = np.empty((T, N1), dtype=np.float32)
    for i in range(N1):
        ids = conn_list[i]
        if ids is None:
            rain_1d[:, i] = global_rain_T
        else:
            rain_1d[:, i] = rain_2d_TN2[:, ids].mean(axis=1)
    return rain_1d


In [6]:

# =============================
# 5) Event dynamic loading to fixed grids
# =============================
os.makedirs(CACHE_DIR, exist_ok=True)

def load_event_arrays(model_id, event_id, split):
    g = graph_cache[model_id]
    N1, N2 = g["N1"], g["N2"]
    idx1, idx2 = g["idx1"], g["idx2"]

    cache_path = os.path.join(CACHE_DIR, f"m{model_id}_{split}_event{event_id}.npz")
    if os.path.exists(cache_path):
        z = np.load(cache_path)
        return z["wl1"], z["wl2"], z["rain2"]

    df1 = load_dynamic_nodes(model_id, event_id, split, 1)
    df2 = load_dynamic_nodes(model_id, event_id, split, 2)
    if df1 is None or df2 is None:
        return None, None, None

    if "water_level" not in df1.columns:
        df1["water_level"] = np.nan
    if "water_level" not in df2.columns:
        df2["water_level"] = np.nan
    if "rainfall" not in df2.columns:
        df2["rainfall"] = 0.0

    T = int(max(df1["t"].max(), df2["t"].max())) + 1

    wl1 = np.full((T, N1), np.nan, dtype=np.float32)
    wl2 = np.full((T, N2), np.nan, dtype=np.float32)
    rain2 = np.full((T, N2), 0.0, dtype=np.float32)

    n = df1["node_idx"].values.astype(int)
    t = df1["t"].values.astype(int)
    w = df1["water_level"].values.astype(np.float32)
    ni = np.array([idx1.get(int(x), -1) for x in n], dtype=np.int64)
    m = ni >= 0
    wl1[t[m], ni[m]] = w[m]

    n = df2["node_idx"].values.astype(int)
    t = df2["t"].values.astype(int)
    w = df2["water_level"].values.astype(np.float32)
    r = df2["rainfall"].values.astype(np.float32)
    ni = np.array([idx2.get(int(x), -1) for x in n], dtype=np.int64)
    m = ni >= 0
    wl2[t[m], ni[m]] = w[m]
    rain2[t[m], ni[m]] = r[m]
    rain2 = np.nan_to_num(rain2, nan=0.0, posinf=0.0, neginf=0.0)

    np.savez_compressed(cache_path, wl1=wl1, wl2=wl2, rain2=rain2)
    return wl1, wl2, rain2


## Node normalization stats (load if cached, else compute once)

In [7]:
# -----------------------------
# 4) Node stats cache helpers
# -----------------------------
def _stats_cache_path(model_id: int, node_type: int) -> Path:
    return CACHE_DIR / f"node_stats_m{model_id}_t{node_type}.npz"

def save_node_stats(model_id: int, node_type: int, st: dict):
    np.savez_compressed(
        _stats_cache_path(model_id, node_type),
        mu=st["mu"].astype(np.float32),
        sd=st["sd"].astype(np.float32),
        gmin=np.float32(st["gmin"]),
        gmax=np.float32(st["gmax"]),
        sd_g=np.float32(st["sd_g"]),
    )

def load_node_stats(model_id: int, node_type: int):
    p = _stats_cache_path(model_id, node_type)
    if not p.exists():
        return None
    z = np.load(p)
    return dict(
        mu=z["mu"].astype(np.float32),
        sd=z["sd"].astype(np.float32),
        gmin=float(z["gmin"]),
        gmax=float(z["gmax"]),
        sd_g=float(z["sd_g"]),
    )

# -----------------------------
# 5) Compute node stats on ALL train events (only if cache missing)
# -----------------------------
def compute_node_stats(model_id: int, node_type: int, train_events):
    g = graph_cache[model_id]
    inv = g["inv1"] if node_type == 1 else g["inv2"]

    sums = None
    sums2 = None
    cnt = None
    gmin = float("inf")
    gmax = float("-inf")

    for ev in train_events:
        wl1, wl2, rain2 = load_event_arrays(model_id, ev, split="train")
        if wl1 is None:
            continue
        Y = wl1 if node_type == 1 else wl2
        Y = np.nan_to_num(Y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

        # stats over all timesteps
        X = Y.reshape(-1, Y.shape[1])
        if sums is None:
            sums = X.sum(axis=0)
            sums2 = (X * X).sum(axis=0)
            cnt = np.full((X.shape[1],), X.shape[0], dtype=np.int64)
        else:
            sums += X.sum(axis=0)
            sums2 += (X * X).sum(axis=0)
            cnt += X.shape[0]

        gmin = min(gmin, float(X.min()))
        gmax = max(gmax, float(X.max()))

    mu = (sums / np.maximum(cnt, 1)).astype(np.float32)
    var = (sums2 / np.maximum(cnt, 1)) - (mu * mu)
    var = np.maximum(var, 1e-6).astype(np.float32)
    sd = np.sqrt(var).astype(np.float32)

    # a global-ish scale for plausible clipping in original units
    sd_g = float(np.mean(sd))

    return dict(mu=mu, sd=sd, gmin=float(gmin), gmax=float(gmax), sd_g=sd_g)

# Build/load stats dict: stats[(model_id, node_type)] = {...}
stats = {}
for model_id in [1, 2]:
    train_events = list_events(model_id, "train")
    for node_type in [1, 2]:
        st = load_node_stats(model_id, node_type)
        if st is None:
            print(f"[stats] computing model={model_id} type={node_type} on {len(train_events)} train events...")
            st = compute_node_stats(model_id, node_type, train_events)
            save_node_stats(model_id, node_type, st)
            print(f"[stats] saved:", _stats_cache_path(model_id, node_type))
        else:
            print(f"[stats] loaded:", _stats_cache_path(model_id, node_type))
        stats[(model_id, node_type)] = st


[stats] computing model=1 type=1 on 68 train events...
[stats] saved: data\_cache_infer\node_stats_m1_t1.npz
[stats] computing model=1 type=2 on 68 train events...
[stats] saved: data\_cache_infer\node_stats_m1_t2.npz
[stats] computing model=2 type=1 on 69 train events...
[stats] saved: data\_cache_infer\node_stats_m2_t1.npz
[stats] computing model=2 type=2 on 69 train events...
[stats] saved: data\_cache_infer\node_stats_m2_t2.npz


## ARX ridge baseline (load if cached, else fit once with lam=0.1)

In [8]:
# =============================
# 7) Baseline ARX ridge fit (per node, global fallback) - simplified
# =============================
def damp_ar_weights(w, n_ar=3, max_sumabs=0.98):
    s = float(np.sum(np.abs(w[:n_ar])))
    if s > max_sumabs:
        w = w.copy()
        w[:n_ar] *= (max_sumabs / s)
    return w

def build_static_by_type(model_id, node_type):
    g = graph_cache[model_id]
    N1, N2 = g["N1"], g["N2"]
    static_all = np.nan_to_num(g["static_all"], nan=0.0, posinf=0.0, neginf=0.0)
    return static_all[:N1] if node_type == 1 else static_all[N1:]

def forward_fill_cols(Y):
    Yf = Y.copy()
    T, N = Yf.shape
    for j in range(N):
        col = Yf[:, j]
        last = np.nan
        for t in range(T):
            if np.isfinite(col[t]):
                last = col[t]
            else:
                col[t] = last
        if not np.isfinite(col[0]):
            col[:] = 0.0
        Yf[:, j] = col
    return Yf

def build_arx_design(Ystd, R, staticX):
    # Ystd: [T,N], R: [T,N], staticX: [N,S]
    T, N = Ystd.shape
    if T <= 3:
        return None, None

    y_t = Ystd[3:T]          # [T-3, N]
    y1  = Ystd[2:T-1]
    y2  = Ystd[1:T-2]
    y3  = Ystd[0:T-3]

    r0 = R[3:T]
    r1 = R[2:T-1]
    r2 = R[1:T-2]
    rc = R[0:T-3] + R[1:T-2] + R[2:T-1]

    feats = [y1, y2, y3, r0, r1, r2, rc]  # 7

    if USE_RAIN_MEMORY:
        R0 = np.nan_to_num(R.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        cum = np.cumsum(R0, axis=0)
        cum_feat = np.clip(cum[2:T-1], 0.0, RAIN_CUM_CLIP)

        ema = np.zeros_like(R0, dtype=np.float32)
        a = float(RAIN_EMA_ALPHA)
        for tt in range(T):
            ema[tt] = a * R0[tt] + (1.0 - a) * (ema[tt-1] if tt > 0 else 0.0)
        ema_feat = ema[2:T-1]

        feats += [cum_feat, ema_feat]  # +2 => 9 total

    base = np.stack(feats, axis=2).astype(np.float32)  # [T-3,N,base_dim]

    if staticX.shape[1] > 0:
        S = staticX.shape[1]
        stat = np.broadcast_to(staticX[None, :, :].astype(np.float32), (T-3, N, S))
        X = np.concatenate([base, stat], axis=2)
    else:
        X = base

    return X, y_t.astype(np.float32)


def solve_ridge_matrix(XTX, XTy, lam):
    d = XTX.shape[0]
    I = np.eye(d, dtype=np.float64)
    try:
        w = np.linalg.solve(XTX + lam * I, XTy)
    except np.linalg.LinAlgError:
        w = np.linalg.lstsq(XTX + lam * I, XTy, rcond=None)[0]
    return damp_ar_weights(w).astype(np.float32)

def fit_baseline(model_id, node_type, events, lam, min_rows_per_node=50):
    g = graph_cache[model_id]
    N1, N2 = g["N1"], g["N2"]
    N = N1 if node_type == 1 else N2

    st = stats[(model_id, node_type)]
    mu = st["mu"][None, :]
    sd = st["sd"][None, :] + 1e-6

    staticX = build_static_by_type(model_id, node_type).astype(np.float32)
    base_dim = 9 if USE_RAIN_MEMORY else 7
    d = base_dim + staticX.shape[1]

    # global accum
    gXTX = np.zeros((d, d), dtype=np.float64)
    gXTy = np.zeros((d,), dtype=np.float64)

    # per node accum (only store for nodes that actually have enough data)
    XTX = [np.zeros((d, d), dtype=np.float64) for _ in range(N)]
    XTy = [np.zeros((d,), dtype=np.float64) for _ in range(N)]
    nrows = np.zeros((N,), dtype=np.int64)

    for ev in events:
        wl1, wl2, rain2 = load_event_arrays(model_id, ev, "train")
        if wl1 is None:
            continue

        if node_type == 1:
            Y = wl1
            global_rain = rain2.mean(axis=1).astype(np.float32)
            R = build_rain_1d_from_2d(rain2, conn_lists[model_id], global_rain)
        else:
            Y = wl2
            R = rain2

        Yf = forward_fill_cols(Y)
        Ystd = (Yf - mu) / sd

        X, yt = build_arx_design(Ystd.astype(np.float32), R.astype(np.float32), staticX)
        if X is None:
            continue

        # accumulate per node and global
        # X: [T-3,N,d], yt: [T-3,N]
        Tm3 = X.shape[0]
        for i in range(N):
            Xi = X[:, i, :].astype(np.float64)     # [Tm3,d]
            yi = yt[:, i].astype(np.float64)       # [Tm3]
            if not np.isfinite(Xi).all() or not np.isfinite(yi).all():
                continue
            XTX[i] += Xi.T @ Xi
            XTy[i] += Xi.T @ yi
            nrows[i] += Tm3

            gXTX += Xi.T @ Xi
            gXTy += Xi.T @ yi

    w_global = solve_ridge_matrix(gXTX, gXTy, lam)

    W_node = np.zeros((N, d), dtype=np.float32)
    for i in range(N):
        if nrows[i] < min_rows_per_node or np.allclose(XTX[i], 0):
            W_node[i] = w_global
        else:
            W_node[i] = solve_ridge_matrix(XTX[i], XTy[i], lam)

    return W_node, w_global, staticX

def baseline_rollout_norm(model_id, node_type, warmup_y, rain_full, W_node, staticX, clip_std=CLIP_STD):
    st = stats[(model_id, node_type)]
    mu = st["mu"][None, :]
    sd = st["sd"][None, :] + 1e-6
    gmin, gmax = st["gmin"], st["gmax"]

    N = warmup_y.shape[1]
    total_T = rain_full.shape[0]
    n_steps = total_T - LOOKBACK

    y_full = np.zeros((total_T, N), dtype=np.float32)
    y_full[:LOOKBACK] = warmup_y.astype(np.float32)

    y_std_hist = np.zeros((total_T, N), dtype=np.float32)
    y_std_hist[:LOOKBACK] = ((y_full[:LOOKBACK] - mu) / sd).astype(np.float32)

    S = staticX.shape[1]
    base_dim = 9 if USE_RAIN_MEMORY else 7
    if USE_RAIN_MEMORY:
        cum_r = np.zeros((N,), dtype=np.float32)
        ema_r = np.zeros((N,), dtype=np.float32)

    for k in range(n_steps):
        t = LOOKBACK + k
        y1 = y_std_hist[t-1]
        y2 = y_std_hist[t-2]
        y3 = y_std_hist[t-3]

        r0 = rain_full[t]
        r1 = rain_full[t-1]
        r2 = rain_full[t-2]
        rc = rain_full[t-1] + rain_full[t-2] + rain_full[t-3]

        feats = [y1, y2, y3, r0, r1, r2, rc]

        if USE_RAIN_MEMORY:
            # update using r1 (rain at t-1) for consistency with design alignment
            r_in = np.nan_to_num(r1.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            cum_r = np.clip(cum_r + r_in, 0.0, RAIN_CUM_CLIP)
            ema_r = (RAIN_EMA_ALPHA * r_in) + ((1.0 - RAIN_EMA_ALPHA) * ema_r)
            feats += [cum_r, ema_r]

        base = np.stack(feats, axis=1).astype(np.float32)  # [N, base_dim]

        X = np.concatenate([base, staticX], axis=1) if S else base

        pred = np.einsum("nd,nd->n", X, W_node)
        pred = np.clip(pred, -clip_std, clip_std).astype(np.float32)

        y_t = (pred[None, :] * sd + mu).astype(np.float32).squeeze(0)
        y_t = np.clip(y_t, gmin - 3.0 * float(st["sd_g"]), gmax + 3.0 * float(st["sd_g"])).astype(np.float32)

        y_full[t] = y_t
        y_std_hist[t] = pred

    return y_full, y_std_hist

# -----------------------------
# Baseline cache helpers
# -----------------------------
def _baseline_cache_path(model_id: int, node_type: int, lam: float) -> Path:
    lam_s = str(lam).replace(".", "p")
    return CACHE_DIR / f"baseline_m{model_id}_t{node_type}_lam{lam_s}.npz"

def save_baseline(model_id: int, node_type: int, lam: float, W_node, w_global, staticX):
    np.savez_compressed(
        _baseline_cache_path(model_id, node_type, lam),
        W_node=W_node.astype(np.float32),
        w_global=w_global.astype(np.float32),
        staticX=staticX.astype(np.float32),
        lam=np.float32(lam),
    )

def load_baseline(model_id: int, node_type: int, lam: float):
    p = _baseline_cache_path(model_id, node_type, lam)
    if not p.exists():
        return None
    z = np.load(p)
    return dict(
        lam=float(z["lam"]),
        W_node=z["W_node"].astype(np.float32),
        w_global=z["w_global"].astype(np.float32),
        staticX=z["staticX"].astype(np.float32),
    )

# -----------------------------
# Fit/load baseline models (no sweep)
# -----------------------------
baseline_models = {}
for model_id in [1, 2]:
    all_events = list_events(model_id, "train")
    for node_type in [1, 2]:
        cached = load_baseline(model_id, node_type, BASELINE_LAM)
        if cached is None:
            print(f"[baseline] fitting model={model_id} type={node_type} lam={BASELINE_LAM} on {len(all_events)} train events...")
            W_node, w_global, staticX = fit_baseline(model_id, node_type, all_events, BASELINE_LAM)
            cached = dict(lam=BASELINE_LAM, W_node=W_node, w_global=w_global, staticX=staticX)
            save_baseline(model_id, node_type, BASELINE_LAM, W_node, w_global, staticX)
            print(f"[baseline] saved:", _baseline_cache_path(model_id, node_type, BASELINE_LAM))
            gc.collect()
        else:
            print(f"[baseline] loaded:", _baseline_cache_path(model_id, node_type, BASELINE_LAM))
        baseline_models[(model_id, node_type)] = cached


[baseline] fitting model=1 type=1 lam=0.1 on 68 train events...
[baseline] saved: data\_cache_infer\baseline_m1_t1_lam0p1.npz
[baseline] fitting model=1 type=2 lam=0.1 on 68 train events...
[baseline] saved: data\_cache_infer\baseline_m1_t2_lam0p1.npz
[baseline] fitting model=2 type=1 lam=0.1 on 69 train events...
[baseline] saved: data\_cache_infer\baseline_m2_t1_lam0p1.npz
[baseline] fitting model=2 type=2 lam=0.1 on 69 train events...
[baseline] saved: data\_cache_infer\baseline_m2_t2_lam0p1.npz


## Correction model definition + checkpoint loading

In [10]:

# =============================
# 8) Coupled correction model
# =============================
def scatter_mean(src_vals, edge_index, num_nodes):
    if edge_index is None:
        return torch.zeros((num_nodes,), device=src_vals.device, dtype=src_vals.dtype)
    src = edge_index[0]
    dst = edge_index[1]
    out = torch.zeros((num_nodes,), device=src_vals.device, dtype=src_vals.dtype)
    cnt = torch.zeros((num_nodes,), device=src_vals.device, dtype=src_vals.dtype)
    out.scatter_add_(0, dst, src_vals[src])
    cnt.scatter_add_(0, dst, torch.ones_like(dst, dtype=src_vals.dtype))
    return out / torch.clamp(cnt, min=1.0)

class CorrectionNet(nn.Module):
    def __init__(self, static_dim, hidden=64):
        super().__init__()
        self.type_emb = nn.Embedding(2, 8)  # 0=1D, 1=2D
        in_dim = 3 + 4 + 1 + 1 + 1 + static_dim + 8  # y_lags + rain + y_base + neigh_same + neigh_cross + static + type_emb
        self.in_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
        )
        self.gru = nn.GRUCell(hidden, hidden)
        self.out_mlp = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )

    def forward_step(self, x_in, h):
        z = self.in_mlp(x_in)
        h = self.gru(z, h)
        raw = self.out_mlp(h).squeeze(1)
        return raw, h

def make_type_ids(N1, N2, device):
    t = torch.zeros((N1 + N2,), device=device, dtype=torch.long)
    t[N1:] = 1
    return t

def teacher_forcing_prob(step):
    if step <= 0:
        return TF_START
    if step >= TF_WARMUP:
        return TF_END
    a = step / float(TF_WARMUP)
    return TF_START + a * (TF_END - TF_START)


# -----------------------------
# Load correction checkpoints
# -----------------------------
def load_corr(model_id: int, ckpt_path: Path):
    static_dim = int(graph_cache[model_id]["static_all"].shape[1])
    model = CorrectionNet(static_dim=static_dim, hidden=CORR_HIDDEN).to(device)
    sd = torch.load(str(ckpt_path), map_location=device)
    model.load_state_dict(sd, strict=True)
    model.eval()
    return model

assert CKPT_CORR_1.exists(), f"missing {CKPT_CORR_1}"
assert CKPT_CORR_2.exists(), f"missing {CKPT_CORR_2}"

corr_models = {
    1: load_corr(1, CKPT_CORR_1),
    2: load_corr(2, CKPT_CORR_2),
}
print("[corr] loaded checkpoints OK")


[corr] loaded checkpoints OK


C:\Users\rowes\AppData\Local\Temp\ipykernel_22452\675846283.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(str(ckpt_path), map_location=device)


In [11]:
def prepare_edges_torch(model_id):
    g = graph_cache[model_id]
    N = g["N"]
    N1 = g["N1"]

    def to_torch(edge):
        if edge is None:
            return None
        return torch.from_numpy(edge).to(device=device, dtype=torch.long)

    e1 = to_torch(g["edges_1d"])
    e2 = to_torch(g["edges_2d"])
    ec = to_torch(g["edges_cross"])

    e_2d1d = None
    e_1d2d = None
    if ec is not None:
        src = ec[0]
        dst = ec[1]
        m_2d1d = (src >= N1) & (dst < N1)
        m_1d2d = (src < N1) & (dst >= N1)
        if m_2d1d.any():
            e_2d1d = torch.stack([src[m_2d1d], dst[m_2d1d]], dim=0)
        if m_1d2d.any():
            e_1d2d = torch.stack([src[m_1d2d], dst[m_1d2d]], dim=0)

    return dict(N=N, N1=N1, e1=e1, e2=e2, e_2d1d=e_2d1d, e_1d2d=e_1d2d)

## Test inference + submission writer

In [12]:

# =============================
# 10) Inference for submission
# =============================
@torch.no_grad()
def predict_event_test(model_id, event_id, sub_event):
    g = graph_cache[model_id]
    edges = prepare_edges_torch(model_id)
    model = corr_models[model_id].to(device).eval()

    N1, N2, N = g["N1"], g["N2"], g["N"]
    inv1, inv2 = g["inv1"], g["inv2"]
    idx1, idx2 = g["idx1"], g["idx2"]

    wl1, wl2, rain2 = load_event_arrays(model_id, event_id, "test")
    if wl1 is None:
        return {}

    T = rain2.shape[0]
    global_rain = rain2.mean(axis=1).astype(np.float32)
    rain1 = build_rain_1d_from_2d(rain2, conn_lists[model_id], global_rain)

    def warmup_from(Y):
        w = Y[:LOOKBACK].copy()
        w[np.isnan(w)] = 0.0
        return w

    warm1 = warmup_from(wl1)
    warm2 = warmup_from(wl2)

    _, base1_std = baseline_rollout_norm(model_id, 1, warm1, rain1[:T],
                                         baseline_models[(model_id,1)]["W_node"],
                                         baseline_models[(model_id,1)]["staticX"])
    _, base2_std = baseline_rollout_norm(model_id, 2, warm2, rain2[:T],
                                         baseline_models[(model_id,2)]["W_node"],
                                         baseline_models[(model_id,2)]["staticX"])

    y_base = torch.from_numpy(np.concatenate([base1_std, base2_std], axis=1).astype(np.float32)).to(device)
    rain_all = torch.from_numpy(np.concatenate([rain1, rain2], axis=1).astype(np.float32)).to(device)

    static_all = torch.from_numpy(g["static_all"]).to(device)
    type_ids = make_type_ids(N1, N2, device)
    type_emb = model.type_emb(type_ids)

    st1 = stats[(model_id, 1)]
    st2 = stats[(model_id, 2)]
    y1_w = (warm1 - st1["mu"][None, :]) / (st1["sd"][None, :] + 1e-6)
    y2_w = (warm2 - st2["mu"][None, :]) / (st2["sd"][None, :] + 1e-6)

    y_pred = torch.zeros((T, N), device=device, dtype=torch.float32)
    y_pred[:LOOKBACK] = torch.from_numpy(np.concatenate([y1_w, y2_w], axis=1).astype(np.float32)).to(device)

    h = torch.zeros((N, HIDDEN), device=device)

    # warm hidden through warmup
    for t in range(3, LOOKBACK):
        y_curr = y_pred[t-1]
        same = torch.zeros((N,), device=device)
        cross = torch.zeros((N,), device=device)
        if edges["e1"] is not None:
            same_1d = scatter_mean(y_curr, edges["e1"], N)
            same[:N1] = same_1d[:N1]
        if edges["e2"] is not None:
            same_2d = scatter_mean(y_curr, edges["e2"], N)
            same[N1:] = same_2d[N1:]
        if edges["e_2d1d"] is not None:
            cross_1d = scatter_mean(y_curr, edges["e_2d1d"], N)
            cross[:N1] = cross_1d[:N1]
        if edges["e_1d2d"] is not None:
            cross_2d = scatter_mean(y_curr, edges["e_1d2d"], N)
            cross[N1:] = cross_2d[N1:]

        y1 = y_pred[t-1]; y2 = y_pred[t-2]; y3 = y_pred[t-3]
        r0 = rain_all[t]; r1 = rain_all[t-1]; r2 = rain_all[t-2]; rc = rain_all[t-1] + rain_all[t-2] + rain_all[t-3]
        x = torch.stack([y1, y2, y3, r0, r1, r2, rc, y_base[t], same, cross], dim=1)
        x = torch.cat([x, static_all, type_emb], dim=1)
        raw, h = model.forward_step(x, h)

    for t in range(LOOKBACK, T):
        y_curr = y_pred[t-1]
        same = torch.zeros((N,), device=device)
        cross = torch.zeros((N,), device=device)
        if edges["e1"] is not None:
            same_1d = scatter_mean(y_curr, edges["e1"], N)
            same[:N1] = same_1d[:N1]
        if edges["e2"] is not None:
            same_2d = scatter_mean(y_curr, edges["e2"], N)
            same[N1:] = same_2d[N1:]
        if edges["e_2d1d"] is not None:
            cross_1d = scatter_mean(y_curr, edges["e_2d1d"], N)
            cross[:N1] = cross_1d[:N1]
        if edges["e_1d2d"] is not None:
            cross_2d = scatter_mean(y_curr, edges["e_1d2d"], N)
            cross[N1:] = cross_2d[N1:]

        y1 = y_pred[t-1]; y2 = y_pred[t-2]; y3 = y_pred[t-3]
        r0 = rain_all[t]; r1 = rain_all[t-1]; r2 = rain_all[t-2]; rc = rain_all[t-1] + rain_all[t-2] + rain_all[t-3]
        x = torch.stack([y1, y2, y3, r0, r1, r2, rc, y_base[t], same, cross], dim=1)
        x = torch.cat([x, static_all, type_emb], dim=1)
        raw, h = model.forward_step(x, h)
        corr = torch.tanh(raw) * CORR_SCALE
        y_pred[t] = torch.clamp(y_base[t] + corr, -CLIP_STD, CLIP_STD)

    y1_pred = (y_pred[:, :N1].cpu().numpy() * (st1["sd"][None, :] + 1e-6) + st1["mu"][None, :]).astype(np.float32)
    y2_pred = (y_pred[:, N1:].cpu().numpy() * (st2["sd"][None, :] + 1e-6) + st2["mu"][None, :]).astype(np.float32)

    out = {}
    for node_type in [1, 2]:
        df_nt = sub_event[sub_event["node_type"] == node_type]
        if df_nt.empty:
            continue
        if node_type == 1:
            y_mat = y1_pred[LOOKBACK:, :]
            idx = idx1
        else:
            y_mat = y2_pred[LOOKBACK:, :]
            idx = idx2

        for node_id, grp in df_nt.groupby("node_id", sort=False):
            node_id = int(node_id)
            n_steps = len(grp)
            j = idx.get(node_id, None)
            if j is None:
                preds = np.zeros((n_steps,), dtype=np.float32)
            else:
                preds_full = y_mat[:, j]
                if len(preds_full) < n_steps:
                    pad = np.full(n_steps - len(preds_full), preds_full[-1] if len(preds_full) else 0.0, dtype=np.float32)
                    preds = np.concatenate([preds_full, pad], axis=0)[:n_steps]
                else:
                    preds = preds_full[:n_steps]
            out[(node_type, node_id)] = preds
    return out

print("Loading sample submission...")
sub = pd.read_csv(SUB_PATH)
sub = sub.drop(columns=["water_level"])
sub["water_level"] = 0.0

print("Running test inference...")
for model_id in sorted(sub["model_id"].unique()):
    df_m = sub[sub["model_id"] == model_id]
    for event_id in sorted(df_m["event_id"].unique()):
        sub_event = df_m[df_m["event_id"] == event_id]
        pred_map = predict_event_test(model_id, event_id, sub_event)

        mask_me = (sub["model_id"] == model_id) & (sub["event_id"] == event_id)
        for (node_type, node_id), preds in pred_map.items():
            m = mask_me & (sub["node_type"] == node_type) & (sub["node_id"] == node_id)
            idxs = sub.index[m]
            n = min(len(idxs), len(preds))
            if n > 0:
                sub.loc[idxs[:n], "water_level"] = preds[:n]

        print(f"[done] model={model_id} event={event_id} filled rows={int(mask_me.sum())}")

sub["water_level"] = sub["water_level"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

out_cols = ["row_id", "model_id", "event_id", "node_type", "node_id", "water_level"]
sub[out_cols].to_csv("fixed_inference.csv", index=False)
print("Wrote fixed_inference.csv")


Loading sample submission...
Running test inference...
[done] model=1 event=5 filled rows=548751
[done] model=1 event=8 filled rows=1444671
[done] model=1 event=18 filled rows=134388
[done] model=1 event=22 filled rows=145587
[done] model=1 event=26 filled rows=134388
[done] model=1 event=29 filled rows=145587
[done] model=1 event=31 filled rows=145587
[done] model=1 event=33 filled rows=548751
[done] model=1 event=35 filled rows=1444671
[done] model=1 event=37 filled rows=145587
[done] model=1 event=42 filled rows=548751
[done] model=1 event=44 filled rows=1444671
[done] model=1 event=48 filled rows=548751
[done] model=1 event=51 filled rows=548751
[done] model=1 event=52 filled rows=1444671
[done] model=1 event=53 filled rows=145587
[done] model=1 event=59 filled rows=145587
[done] model=1 event=62 filled rows=548751
[done] model=1 event=65 filled rows=1444671
[done] model=1 event=66 filled rows=134388
[done] model=1 event=67 filled rows=548751
[done] model=1 event=69 filled rows=145